[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S11/2_Practica/para_casa/S11_03_deteccion_segmentacion.ipynb)

# S11 · 03 — Detección de objetos y segmentación

Sprint 11 · UD 4.6

**Contexto TurisData.** Clasificar una foto dice *qué es*; pero al cliente le interesan preguntas como *"¿cuántas sombrillas hay en la playa del resort?"* (detección) o *"¿qué parte de la imagen es piscina?"* (segmentación). En esta práctica verás los conceptos, las métricas (IoU, NMS, mAP) con números pequeños y un ejemplo completo **sin internet**, con imágenes aéreas sintéticas de una playa y con OpenCV.

**Este cuaderno tiene 5 ejercicios y un mini-reto.** Los ejercicios 1 y 2 son los mismos que los del cuaderno «en clase» (allí, del 1 al 2).

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import os, sys, subprocess
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageDraw

try:
    import cv2
except ImportError:      # en Colab y en el taller ya viene instalado; esta celda solo actúa si faltara
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless"])
    import cv2

SEMILLA = 42
print("OpenCV", cv2.__version__)
print("Listo ✔")

### ✏️ A2 · EJERCICIO 1 de 5 — La IoU de dos cajas

**Tu turno:** Escribe `iou(a, b)` que reciba dos cajas `[x1, y1, x2, y2]` y devuelva su IoU (un número entre 0 y 1). Cuidado con el caso en que no se solapan (la intersección vale 0, no negativa).

**Debe salir:** ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A2).

1. ¿Cuánto vale la IoU de `[0,0,10,10]` y `[5,5,15,15]`?
2. ¿Por qué se acota con `max(0, ...)`?
3. ¿Cuándo se considera correcta una detección?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A2, **después de intentarlo**.

In [ ]:
# pista: ancho y alto de la intersección con max(0, ...); unión = área_a + área_b - intersección
def iou(a, b):
    ...

assert abs(iou([0, 0, 10, 10], [5, 5, 15, 15]) - 25 / 175) < 1e-9, "IoU esperada 25/175 ≈ 0.143"
assert iou([0, 0, 10, 10], [0, 0, 10, 10]) == 1.0, "Cajas idénticas: IoU = 1"
assert iou([0, 0, 10, 10], [20, 20, 30, 30]) == 0.0, "Cajas separadas: IoU = 0 (¿has acotado con max(0, ...)?)"
assert abs(iou([0, 0, 10, 10], [0, 0, 10, 5]) - 0.5) < 1e-9, "Una caja dentro de otra: IoU = área pequeña / área grande"
print("✔ Ejercicio 1 correcto")

### ✏️ A3 · EJERCICIO 2 de 5 — NMS: quedarse con una caja por objeto

**Tu turno:** Escribe `nms(cajas, puntuaciones, umbral_iou=0.5)` que devuelva la **lista de índices** de las cajas que se conservan, en orden de confianza decreciente. Usa tu función `iou`.

**Debe salir:** ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A3).

1. ¿Qué cajas conserva NMS en el ejemplo con umbral 0,5?
2. ¿Qué pasa con un umbral de 0,99?
3. ¿Por qué un detector propone cajas duplicadas?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A3, **después de intentarlo**.

In [ ]:
# pista: ordena índices por puntuación; mientras queden, guarda el primero y filtra los que tienen iou > umbral con él
def nms(cajas, puntuaciones, umbral_iou=0.5):
    ...

_cajas = [[10, 10, 50, 50], [12, 12, 52, 52], [100, 100, 140, 140], [11, 9, 49, 51]]
_punt = [0.9, 0.8, 0.7, 0.6]
assert nms(_cajas, _punt) == [0, 2], "Debe quedar la caja 0 (mejor) y la 2 (lejana); la 1 y la 3 son duplicados de la 0"
assert nms(_cajas, _punt, umbral_iou=0.99) == [0, 1, 2, 3], "Con un umbral muy alto no se descarta nada"
assert nms([[0, 0, 5, 5]], [0.3]) == [0], "Con una sola caja se conserva"
print("✔ Ejercicio 2 correcto")

### ✏️ A4 · EJERCICIO 3 de 5 — El AP (average precision)

**Tu turno:** Escribe `average_precision(es_tp, n_reales)`: recibe una lista de 0/1 con las detecciones **ya ordenadas por confianza decreciente** (1 = verdadero positivo) y el número de objetos reales `n_reales`, y devuelve el AP. Pasos: acumula TP y FP; calcula `precision = TP/(TP+FP)` y `recall = TP/n_reales` en cada puesto; haz la precisión monótona (de derecha a izquierda, cada valor pasa a ser el máximo de los que tiene a su derecha); suma `(recall_i − recall_{i−1}) × precision_i`.

**Debe salir:** ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A4).

1. ¿Cuánto vale el AP de la tabla del enunciado?
2. ¿Por qué la 5.ª detección no añade nada al AP?
3. ¿Qué mide el mAP?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A4, **después de intentarlo**.

In [ ]:
# pista: np.cumsum para TP y FP; la precisión se "suaviza" de derecha a izquierda; suma ponderada por el aumento de recall
def average_precision(es_tp, n_reales):
    ...

assert abs(average_precision([1, 1, 0, 1, 0], 4) - 0.6875) < 1e-9, "Con la tabla del enunciado el AP debe ser 0.6875"
assert abs(average_precision([1, 1, 1, 1], 4) - 1.0) < 1e-9, "Detector perfecto: AP = 1"
assert abs(average_precision([1, 1], 4) - 0.5) < 1e-9, "Solo encuentra la mitad de los objetos: AP = 0.5"
assert average_precision([0, 0, 0], 3) == 0.0, "Sin ningún acierto el AP es 0"
print("✔ Ejercicio 3 correcto")

### 👀 A5 · EJEMPLO — Generar una escena de playa con sombrillas

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A5).

1. ¿Cuántas sombrillas tiene la primera escena?
2. ¿Qué devuelve `escena` además de la imagen?
3. ¿Para qué sirven los distractores (palmeras, tumbonas)?

In [ ]:
def _col(rng, base, var=12):
    return tuple(int(np.clip(c + rng.integers(-var, var + 1), 0, 255)) for c in base)

def escena(rng, tam=128, n_max=8):
    """Devuelve (imagen RGB uint8, cajas reales [x1,y1,x2,y2], máscara booleana de sombrillas)."""
    img = Image.new("RGB", (tam, tam), _col(rng, (230, 210, 160)))
    d = ImageDraw.Draw(img)
    mar = int(tam * rng.uniform(0.15, 0.28))
    d.rectangle([0, 0, tam, mar], fill=_col(rng, (60, 140, 200), 15))
    masc = Image.new("L", (tam, tam), 0)
    dm = ImageDraw.Draw(masc)
    ocupados, cajas = [], []
    n = int(rng.integers(3, n_max + 1))
    for _ in range(300):
        if len(cajas) >= n + int(rng.integers(2, 5)) and len(cajas) >= n:
            break
        r = int(rng.integers(6, 10))
        cx, cy = int(rng.integers(r + 2, tam - r - 2)), int(rng.integers(mar + r + 3, tam - r - 2))
        if any(np.hypot(cx - x, cy - y) < r + rr - 4 for x, y, rr, _ in ocupados):   # se permite algo de solape
            continue
        if len([o for o in ocupados if o[3] == "s"]) < n:                      # sombrilla
            color = [(220, 40, 40), (240, 140, 30), (240, 210, 40), (245, 225, 140)][int(rng.integers(0, 4))]   # la 4.ª es pastel (poco saturada)
            d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=_col(rng, color, 10))
            d.ellipse([cx - 1, cy - 1, cx + 1, cy + 1], fill=(90, 60, 40))
            dm.ellipse([cx - r, cy - r, cx + r, cy + r], fill=255)
            cajas.append([cx - r, cy - r, cx + r, cy + r]); ocupados.append((cx, cy, r, "s"))
        elif rng.random() < 0.5:                                                # palmera (distractor verde)
            d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=_col(rng, (30, 140, 60), 15)); ocupados.append((cx, cy, r, "p"))
        else:                                                                   # tumbona (distractor gris)
            d.rectangle([cx - r, cy - 3, cx + r, cy + 3], fill=_col(rng, (150, 150, 155), 15)); ocupados.append((cx, cy, r, "t"))
    a = np.asarray(img, dtype="float32") + rng.normal(0, 6, (tam, tam, 3))
    return np.clip(a, 0, 255).astype("uint8"), np.array(cajas), np.asarray(masc) > 0

def dibujar_cajas(eje, cajas, color, etiquetas=None):
    for k, (x1, y1, x2, y2) in enumerate(cajas):
        eje.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, lw=1.6))
        if etiquetas is not None:
            eje.text(x1, y1 - 1, etiquetas[k], color=color, fontsize=7, va="bottom")

rng = np.random.default_rng(SEMILLA)
img0, cajas0, masc0 = escena(rng)
fig, ejes = plt.subplots(1, 2, figsize=(9, 4.4))
ejes[0].imshow(img0); ejes[0].set_title("Escena sintética"); ejes[0].axis("off")
ejes[1].imshow(img0); dibujar_cajas(ejes[1], cajas0, "lime"); ejes[1].set_title(f"Cajas reales ({len(cajas0)} sombrillas)"); ejes[1].axis("off")
plt.tight_layout(); plt.show()

### 👀 A6 · EJEMPLO — Detector 1: template matching + NMS

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A6).

1. ¿Cuántas detecciones hay antes y después de NMS?
2. ¿Por qué se trabaja sobre el canal de saturación?
3. ¿Qué muestra este ejemplo sobre el uso de NMS?

In [ ]:
def saturacion(img_rgb):
    return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)[:, :, 1]

plantilla_img = Image.new("RGB", (24, 24), (230, 210, 160))
ImageDraw.Draw(plantilla_img).ellipse([4, 4, 20, 20], fill=(230, 100, 30))
plantilla = saturacion(np.asarray(plantilla_img))
LADO = 16    # caja que asignamos a cada acierto (diámetro aproximado de una sombrilla)

def detectar_plantilla(img_rgb, umbral=0.6):
    res = cv2.matchTemplate(saturacion(img_rgb), plantilla, cv2.TM_CCOEFF_NORMED)
    ys, xs = np.where(res >= umbral)
    cajas = [[x + 4, y + 4, x + 4 + LADO, y + 4 + LADO] for x, y in zip(xs, ys)]      # el círculo está a 4 px del borde de la plantilla
    return cajas, [float(res[y, x]) for x, y in zip(xs, ys)]

cajas_brutas, punt_brutas = detectar_plantilla(img0)
quedan = nms(cajas_brutas, punt_brutas, umbral_iou=0.3)
print(f"Detecciones brutas: {len(cajas_brutas)} → tras NMS: {len(quedan)} (sombrillas reales: {len(cajas0)})")

fig, ejes = plt.subplots(1, 2, figsize=(9, 4.4))
for eje, cajas, titulo in [(ejes[0], cajas_brutas, f"Antes de NMS ({len(cajas_brutas)} cajas)"),
                           (ejes[1], [cajas_brutas[i] for i in quedan], f"Después de NMS ({len(quedan)} cajas)")]:
    eje.imshow(img0); dibujar_cajas(eje, cajas, "red"); dibujar_cajas(eje, cajas0, "lime"); eje.set_title(titulo); eje.axis("off")
plt.suptitle("Rojo = detecciones · verde = cajas reales"); plt.tight_layout(); plt.show()

### 👀 A7 · EJEMPLO — Detector 2: color + contornos

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A7).

1. ¿Qué hace la apertura morfológica?
2. ¿Cómo se calcula la confianza de cada caja?
3. ¿Cuántas cajas propone para la primera escena y cuántas sombrillas hay?

In [ ]:
def detectar_color(img_rgb, sat_min=120, area_min=40):
    """Devuelve (cajas, confianzas, máscara) de zonas de color vivo rojo–naranja–amarillo."""
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    h, s, v = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]
    mascara = (((h <= 35) | (h >= 170)) & (s >= sat_min) & (v >= 120)).astype("uint8") * 255
    mascara = cv2.morphologyEx(mascara, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    contornos, _ = cv2.findContours(mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cajas, conf = [], []
    for c in contornos:
        area = cv2.contourArea(c)
        if area < area_min:
            continue
        x, y, w, h_ = cv2.boundingRect(c)
        cajas.append([x, y, x + w, y + h_])
        conf.append(float(min(1.0, area / (np.pi / 4 * w * h_))))          # 1 = perfectamente circular/relleno
    return cajas, conf, mascara > 0

cajas_c, conf_c, masc_c = detectar_color(img0)
fig, ejes = plt.subplots(1, 3, figsize=(13, 4.2))
ejes[0].imshow(img0); ejes[0].set_title("Imagen"); ejes[0].axis("off")
ejes[1].imshow(masc_c, cmap="gray"); ejes[1].set_title("Máscara de color vivo (tras apertura)"); ejes[1].axis("off")
ejes[2].imshow(img0); dibujar_cajas(ejes[2], cajas_c, "red", [f"{c:.2f}" for c in conf_c]); dibujar_cajas(ejes[2], cajas0, "lime")
ejes[2].set_title("Cajas del detector (rojo) y reales (verde)"); ejes[2].axis("off")
plt.tight_layout(); plt.show()

### 👀 A8 · EJEMPLO — Evaluar el detector: emparejar con la realidad

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A8).

1. ¿Qué precisión y qué recall tiene el detector de color?
2. ¿Qué significa ese recall?
3. ¿Por qué una caja real no puede emparejarse dos veces?

In [ ]:
def emparejar(cajas_det, conf, cajas_reales, umbral_iou=0.5):
    """Devuelve la lista de 0/1 (TP/FP) de las detecciones ordenadas por confianza decreciente."""
    orden = np.argsort(conf)[::-1]
    usadas, es_tp = set(), []
    for i in orden:
        mejor, mejor_iou = None, umbral_iou
        for j, real in enumerate(cajas_reales):
            u = iou(cajas_det[i], real)
            if j not in usadas and u >= mejor_iou:
                mejor, mejor_iou = j, u
        if mejor is not None:
            usadas.add(mejor); es_tp.append(1)
        else:
            es_tp.append(0)
    return es_tp

def evaluar_detector(detector, n_escenas=30, semilla=100, umbral_iou=0.5):
    """AP, precisión y recall globales sobre n_escenas nuevas. `detector(img)` devuelve (cajas, confianzas)."""
    rng = np.random.default_rng(semilla)
    todos_conf, todos_tp, n_reales, n_detectadas = [], [], 0, 0
    for _ in range(n_escenas):
        img, reales, _ = escena(rng)
        cajas, conf = detector(img)
        n_reales += len(reales); n_detectadas += len(cajas)
        tp = emparejar(cajas, conf, reales, umbral_iou) if len(cajas) else []
        orden = np.argsort(conf)[::-1] if len(cajas) else []
        todos_conf += [conf[i] for i in orden]; todos_tp += tp
    orden = np.argsort(todos_conf)[::-1]
    tp_global = [todos_tp[i] for i in orden]
    return {"AP": average_precision(tp_global, n_reales),
            "precisión": sum(tp_global) / max(1, n_detectadas), "recall": sum(tp_global) / n_reales,
            "reales": n_reales, "detectadas": n_detectadas}

def detector_color(img):
    cajas, conf, _ = detectar_color(img)
    return cajas, conf

resultado = evaluar_detector(detector_color)
print({k: round(v, 3) if isinstance(v, float) else v for k, v in resultado.items()})

### ✏️ A9 · EJERCICIO 4 de 5 — El mAP según el umbral de IoU

**Tu turno:** Calcula el **mAP@[.5:.9]** simplificado del detector de color: la media del AP con umbrales de IoU `0.5, 0.6, 0.7, 0.8, 0.9` (usa `evaluar_detector(detector_color, umbral_iou=u)["AP"]`). Guarda la media en `map_iou` y, en `ap_por_umbral`, la lista de los 5 AP. Como solo hay una clase (sombrilla), el mAP coincide con el AP.

**Debe salir:** una curva del AP frente al umbral de IoU y ✔ Ejercicio 4 correcto (AP ≈ 0.54 · 0.51 · 0.48 · 0.29 · 0.00; media ≈ 0.36)

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A9).

1. ¿Cuánto vale el mAP del detector de color con estos umbrales?
2. ¿Por qué el AP baja al subir el umbral de IoU?
3. ¿Qué indica un AP casi 0 con IoU ≥ 0,9?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A9, **después de intentarlo**.

In [ ]:
umbrales = [0.5, 0.6, 0.7, 0.8, 0.9]      # los umbrales de IoU que vas a probar (ya escritos)
# pista: un bucle sobre los umbrales llamando a evaluar_detector; np.mean de la lista
ap_por_umbral = ...
map_iou = ...

assert len(ap_por_umbral) == 5, "Debe haber un AP por cada umbral (5 en total)"
assert all(ap_por_umbral[i] >= ap_por_umbral[i + 1] - 1e-9 for i in range(4)), "El AP no debería subir al exigir más IoU. Revisa el orden de los umbrales"
assert 0 <= map_iou <= 1, "El mAP es un número entre 0 y 1"

plt.plot(umbrales, ap_por_umbral, "o-"); plt.xlabel("umbral de IoU exigido"); plt.ylabel("AP")
plt.title(f"Detector de color: AP según el umbral de IoU (media = {map_iou:.2f})"); plt.ylim(0, 1.02); plt.show()
print("✔ Ejercicio 4 correcto")

### ✏️ A10 · EJERCICIO 5 de 5 — IoU y Dice de máscaras

**Tu turno:** Escribe `iou_mascara(pred, real)` y `dice(pred, real)` para máscaras booleanas del mismo tamaño (`True` = sombrilla). Devuelve 1.0 si ambas están vacías.

**Debe salir:** ✔ Ejercicio 5 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A10).

1. ¿Cuánto valen IoU y Dice de `_p` y `_r`?
2. ¿Por qué se devuelve 1.0 si ambas máscaras están vacías?
3. ¿Qué diferencia hay entre IoU y Dice?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A10, **después de intentarlo**.

In [ ]:
# pista: usa np.logical_and / np.logical_or y suma los True (.sum())
def iou_mascara(pred, real):
    ...

def dice(pred, real):
    ...

_p = np.array([[1, 1, 0], [0, 0, 0]], dtype=bool); _r = np.array([[1, 0, 0], [1, 0, 0]], dtype=bool)
assert abs(iou_mascara(_p, _r) - 1 / 3) < 1e-9, "Intersección 1, unión 3 → 1/3"
assert abs(dice(_p, _r) - 0.5) < 1e-9, "2·1 / (2 + 2) = 0.5"
assert iou_mascara(np.zeros((2, 2), bool), np.zeros((2, 2), bool)) == 1.0, "Dos máscaras vacías coinciden: 1.0"
print("✔ Ejercicio 5 correcto")

### 👀 A11 · EJEMPLO — Segmentar con el umbral de color

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A11).

1. ¿Qué IoU y Dice medios logra la segmentación por color?
2. ¿Por qué el Dice sale mayor que la IoU?
3. ¿Hace falta entrenar algo para obtener esa máscara?

In [ ]:
rng = np.random.default_rng(200)
ious_umbral = []
for _ in range(20):
    im, _, real = escena(rng)
    _, _, pred = detectar_color(im)
    ious_umbral.append((iou_mascara(pred, real), dice(pred, real)))
print("Segmentación por umbral de color: IoU media =", round(np.mean([a for a, _ in ious_umbral]), 3),
      "· Dice medio =", round(np.mean([b for _, b in ious_umbral]), 3))

### 👀 A12 · EJEMPLO — Segmentar por agrupación de colores (k-means)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A12).

1. ¿Cuántos colores quedan en la imagen segmentada?
2. ¿Sabe k-means qué es cada zona (flor, hoja…)?
3. ¿Cuándo funciona bien esta idea?

In [ ]:
from sklearn.datasets import load_sample_images
foto = load_sample_images().images[1]                                # flor (427×640×3)
peq = cv2.resize(foto, (160, 107)).reshape(-1, 3).astype("float32")
criterio = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
cv2.setRNGSeed(SEMILLA)
_, etiquetas, centros = cv2.kmeans(peq, 4, None, criterio, 3, cv2.KMEANS_PP_CENTERS)
segmentada = centros.astype("uint8")[etiquetas.ravel()].reshape(107, 160, 3)
fig, ejes = plt.subplots(1, 2, figsize=(10, 3.6))
ejes[0].imshow(foto); ejes[0].set_title("Foto original"); ejes[0].axis("off")
ejes[1].imshow(segmentada); ejes[1].set_title("Segmentada con k-means (4 colores)"); ejes[1].axis("off")
plt.tight_layout(); plt.show()

### 👀 A13 · EJEMPLO — Una mini red de segmentación (encoder-decoder)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A13).

1. ¿Qué IoU logran la mini red y el umbral de color?
2. ¿Qué hace cada mitad de una red encoder-decoder?
3. ¿Cuándo preferirías la red al umbral de color?

In [ ]:
import tensorflow as tf
from tensorflow import keras
keras.utils.set_random_seed(SEMILLA)

rng = np.random.default_rng(300)
datos = [escena(rng) for _ in range(360)]
X_seg = np.stack([d[0] for d in datos]).astype("float32") / 255
Y_seg = np.stack([d[2] for d in datos]).astype("float32")[..., None]
X_s_ent, Y_s_ent, X_s_pru, Y_s_pru = X_seg[:300], Y_seg[:300], X_seg[300:], Y_seg[300:]

def construir_segmentador():
    e = keras.layers.Input(shape=(128, 128, 3))
    x = keras.layers.Conv2D(8, 3, padding="same", activation="relu")(e)
    x = keras.layers.MaxPooling2D()(x)                                         # 64×64
    x = keras.layers.Conv2D(16, 3, padding="same", activation="relu")(x)
    x = keras.layers.UpSampling2D()(x)                                         # 128×128
    x = keras.layers.Conv2D(8, 3, padding="same", activation="relu")(x)
    s = keras.layers.Conv2D(1, 1, activation="sigmoid")(x)                      # probabilidad por píxel
    m = keras.Model(e, s)
    m.compile("adam", "binary_crossentropy")
    return m

seg = construir_segmentador()
h_seg = seg.fit(X_s_ent, Y_s_ent, epochs=12, batch_size=16, validation_split=0.1, verbose=0)
pred_seg = seg.predict(X_s_pru, verbose=0)[..., 0] > 0.5
iou_red = np.mean([iou_mascara(p, r[..., 0] > 0) for p, r in zip(pred_seg, Y_s_pru)])
iou_umb = np.mean([iou_mascara(detectar_color((x * 255).astype("uint8"))[2], r[..., 0] > 0) for x, r in zip(X_s_pru, Y_s_pru)])
print(f"IoU de máscaras en 60 escenas nuevas → mini red: {iou_red:.3f} · umbral de color: {iou_umb:.3f}")

fig, ejes = plt.subplots(1, 3, figsize=(11, 3.8))
ejes[0].imshow(X_s_pru[0]); ejes[0].set_title("Imagen"); ejes[1].imshow(Y_s_pru[0, ..., 0], cmap="gray"); ejes[1].set_title("Máscara real")
ejes[2].imshow(pred_seg[0], cmap="gray"); ejes[2].set_title("Predicción de la red")
for e in ejes: e.axis("off")
plt.tight_layout(); plt.show()

### 👀 A14 · EJEMPLO — Un detector moderno con YOLO (opcional, requiere internet)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A14).

1. ¿Qué devuelve YOLO por cada objeto?
2. ¿Qué parte de lo que has programado a mano aparece dentro de YOLO?
3. ¿Por qué etiquetar para un detector cuesta más que para un clasificador?

In [ ]:
USAR_INTERNET = False        # ← True solo con conexión (Colab)

if USAR_INTERNET:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
    from ultralytics import YOLO
    modelo_yolo = YOLO("yolov8n.pt")                                                     # descarga los pesos
    resultados = modelo_yolo("https://ultralytics.com/images/bus.jpg", conf=0.25, iou=0.5)    # descarga la imagen
    r = resultados[0]
    plt.imshow(r.plot()[:, :, ::-1]); plt.axis("off"); plt.title("YOLOv8n (COCO)"); plt.show()
    for caja, cls, c in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist()):
        print(r.names[int(cls)], round(c, 2), [round(v) for v in caja])
    # Para tu dataset propio: modelo_yolo.train(data="datos.yaml", epochs=30, imgsz=640)  (requiere etiquetas en formato YOLO)
else:
    print("Rama de Colab desactivada (USAR_INTERNET = False). El resto del notebook no la necesita.")

### ✏️ A15 · MINI-RETO — Contador de sombrillas

**Tu turno:** El cliente quiere un **contador de sombrillas** por foto de la playa. Con 30 escenas nuevas (`semilla=500`): 1. Calcula en `error_conteo` el error absoluto medio entre el número de cajas del `detector_color` y el número real. 2. Prueba `sat_min` ∈ {60, 90, 120, 150, 180} y guarda en `mejor_sat` el valor que da mayor **F1** de detección (F1 = 2·P·R/(P+R), con IoU ≥ 0,5, usando `evaluar_detector` con un detector que fije ese `sat_min`) y en `f1_mejor` su valor. Con sat_min bajo la arena "cuela" como sombrilla; con sat_min alto se pierden sombrillas amarillas. Después, responde con tus palabras (🧠).

**Debe salir:** `Error medio de conteo: 1.87 sombrillas por foto · mejor sat_min = 90 (F1 = 0.71)`, el diccionario de F1 por `sat_min` y ✔ Mini-reto correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A15).

1. ¿Qué `sat_min` da el mejor F1 y cuánto vale?
2. ¿Por qué falla con `sat_min = 60`?
3. ¿Por qué el F1 no cambia entre 120 y 180?
4. ¿Qué pasaría con sombrillas blancas?
5. ¿Qué harías para que funcione con fotos reales?

In [ ]:
# pista: cuenta len(cajas) frente a len(reales) por escena; para el F1 usa functools.partial o una lambda con sat_min
error_conteo = ...
f1_por_sat = {}
mejor_sat = ...
f1_mejor = ...

assert error_conteo >= 0, "El error de conteo no puede ser negativo"
assert mejor_sat in (60, 90, 120, 150, 180), "mejor_sat debe ser uno de los valores probados"
assert f1_mejor > 0.5, "Con el mejor umbral el F1 debería superar 0.5. Revisa cómo calculas precisión y recall"
print(f"Error medio de conteo: {error_conteo:.2f} sombrillas por foto · mejor sat_min = {mejor_sat} (F1 = {f1_mejor:.2f})")
print({s: round(v, 2) for s, v in f1_por_sat.items()})
print("✔ Mini-reto correcto")

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas

- **Qué hace:**
    - Carga `numpy`, `matplotlib`, `PIL` (para dibujar imágenes) y `cv2` (OpenCV, la librería de visión clásica). Si `cv2` faltara, la celda lo instala.
    - `os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"` silencia los mensajes técnicos de TensorFlow, que se usará más adelante.
- **Qué debe salir:**
    - `OpenCV x.y.z` (la versión instalada, por ejemplo 5.0.0) y `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'np' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 2. Cajas y la IoU

#### A2 · Ejercicio 1 — La IoU de dos cajas

**Contexto TurisData.** Clasificar una foto dice *qué es*; pero al cliente le interesan preguntas como *«¿cuántas sombrillas hay en la playa del resort?»* (detección) o *«¿qué parte de la imagen es piscina?»* (segmentación). En esta práctica verás los conceptos, las métricas (IoU, NMS, mAP) con números pequeños y un ejemplo completo **sin internet**, con imágenes aéreas sintéticas de una playa y con OpenCV.

| Tarea | Pregunta | Salida | Ejemplo TurisData |
|---|---|---|---|
| **Clasificación** | ¿Qué hay en la foto? | una etiqueta | "esta foto es de una piscina" |
| **Detección** | ¿Qué hay y **dónde**? | cajas (x1, y1, x2, y2) + clase + confianza | contar sombrillas en la playa |
| **Segmentación semántica** | ¿A qué clase pertenece **cada píxel**? | una máscara | superficie de piscina en la foto |
| **Segmentación de instancias** | Lo mismo, separando cada objeto | una máscara por objeto | cada tumbona por separado |

Los detectores modernos (YOLO, Faster R-CNN, DETR) son redes convolucionales que predicen muchas cajas candidatas de una vez;
después se limpian con **NMS** y se evalúan con **mAP**. Empezamos por esas piezas.

**Idea.** Una caja se escribe `[x1, y1, x2, y2]` (esquina superior izquierda y esquina inferior derecha, en píxeles).
Para saber si una caja predicha "acierta" con la real, se mide cuánto se solapan:

`IoU = área de la intersección / área de la unión`

0 = no se tocan; 1 = idénticas. Por convención, una detección es correcta (verdadero positivo) si su IoU con una caja real es **≥ 0,5**.

**Ejemplo:** cajas `[0,0,10,10]` y `[5,5,15,15]`: intersección 5×5 = 25; unión 100 + 100 − 25 = 175; IoU = 25/175 ≈ 0,143.

- **Qué hace:**
    - `ancho` y `alto` son las dimensiones de la **intersección** de las dos cajas: `min` de los extremos derechos menos `max` de los izquierdos (y lo mismo en vertical), acotados con `max(0, ...)` para que no salgan negativos si las cajas no se tocan.
    - `inter` es el área de la intersección; `area_a` y `area_b` las de cada caja; la unión es `area_a + area_b - inter`.
    - `iou` devuelve `inter / union` (o 0 si la unión es 0). Las comprobaciones prueban solape parcial (25/175 ≈ 0,143), cajas idénticas (1), cajas separadas (0) y una caja dentro de otra (0,5).
- **Qué debe salir:**
    - `✔ Ejercicio 1 correcto`.
- **Cómo leerlo:**
    - La IoU va de 0 (no se tocan) a 1 (idénticas). Por convención, una detección es correcta (verdadero positivo) si su IoU con una caja real es **≥ 0,5**.
- **Si falla:**
    - `IoU esperada 25/175 ≈ 0.143` → la unión o la intersección están mal calculadas (recuerda restar la intersección de la suma de áreas).
    - `Cajas separadas: IoU = 0 (¿has acotado con max(0, ...)?)` → falta acotar `ancho` y `alto` con `max(0, ...)`.
    - `NameError: name 'iou' is not defined` → no has ejecutado la celda del ejercicio.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto vale la IoU de `[0,0,10,10]` y `[5,5,15,15]`?* → **25 / 175 ≈ 0,143**: intersección 5×5 = 25 y unión 100 + 100 − 25 = 175.
2. *¿Por qué se acota con `max(0, ...)`?* → Para que, si las cajas **no se solapan**, la intersección valga 0 y no un número negativo.
3. *¿Cuándo se considera correcta una detección?* → Cuando su IoU con una caja real es **≥ 0,5**.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def iou(a, b):
    ancho = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    alto = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ancho * alto
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 3. NMS: una caja por objeto

#### A3 · Ejercicio 2 — NMS: quedarse con una caja por objeto

**Idea.** Un detector suele proponer **varias cajas casi iguales** para el mismo objeto. NMS las limpia:
1. Ordena las cajas por confianza (de mayor a menor).
2. Se queda con la primera.
3. **Descarta** las demás que se solapan con ella más de un umbral (por ejemplo IoU > 0,5).
4. Repite con las que quedan.

**Ejemplo:** cajas A (confianza 0,9), B (0,8, casi igual a A) y C (0,7, lejos). NMS conserva A y C: B era un duplicado de A.

- **Qué hace:**
    - `np.argsort(puntuaciones)[::-1]` ordena los índices de las cajas por confianza, de mayor a menor (`orden`).
    - Mientras queden índices, `orden.pop(0)` toma el de mayor confianza (`mejor`), lo guarda en `conservadas` y se queda en `orden` solo con las cajas cuyo `iou` con `mejor` es **menor o igual** que `umbral_iou`.
    - Las comprobaciones usan 4 cajas: la 0 (confianza 0,9), la 1 y la 3 (casi iguales a la 0) y la 2 (lejana).
- **Qué debe salir:**
    - `✔ Ejercicio 2 correcto`.
- **Cómo leerlo:**
    - Con el umbral de 0,5 se conservan las cajas `[0, 2]`: la 1 y la 3 eran duplicados de la 0. Con un umbral de 0,99 no se descarta ninguna: `[0, 1, 2, 3]`.
- **Si falla:**
    - `Debe quedar la caja 0 (mejor) y la 2 (lejana)...` → no filtras las cajas que se solapan con la elegida, o filtras con la condición al revés.
    - `Con un umbral muy alto no se descarta nada` → el umbral no se usa o se compara mal (`<=` frente a `<`).
    - `NameError: name 'iou' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué cajas conserva NMS en el ejemplo con umbral 0,5?* → **[0, 2]**: la mejor y la lejana.
2. *¿Qué pasa con un umbral de 0,99?* → Casi nada se considera duplicado: se conservan las **4** cajas.
3. *¿Por qué un detector propone cajas duplicadas?* → Porque predice **muchas cajas candidatas** a la vez, y varias caen casi en el mismo objeto.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def nms(cajas, puntuaciones, umbral_iou=0.5):
    orden = list(np.argsort(puntuaciones)[::-1])
    conservadas = []
    while orden:
        mejor = orden.pop(0)
        conservadas.append(int(mejor))
        orden = [i for i in orden if iou(cajas[mejor], cajas[i]) <= umbral_iou]
    return conservadas
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 4. Métricas: precisión, recall, AP y mAP

#### A4 · Ejercicio 3 — El AP (average precision)

Imagina que en una foto hay **4 sombrillas reales** y el detector entrega 5 cajas con su confianza. Tras ordenarlas por
confianza y compararlas con la realidad (IoU ≥ 0,5), quedan así (✔ = verdadero positivo, ✘ = falso positivo):

| Puesto | Confianza | ¿Correcta? | TP acumulados | Precisión acumulada | Recall acumulado |
|---|---|---|---|---|---|
| 1 | 0,95 | ✔ | 1 | 1/1 = 1,00 | 1/4 = 0,25 |
| 2 | 0,90 | ✔ | 2 | 2/2 = 1,00 | 2/4 = 0,50 |
| 3 | 0,80 | ✘ | 2 | 2/3 = 0,67 | 2/4 = 0,50 |
| 4 | 0,70 | ✔ | 3 | 3/4 = 0,75 | 3/4 = 0,75 |
| 5 | 0,60 | ✘ | 3 | 3/5 = 0,60 | 3/4 = 0,75 |

- **Precisión:** de lo que el detector dice, ¿cuánto es cierto? **Recall:** de lo que hay, ¿cuánto encuentra?
- Al bajar el umbral de confianza suben los aciertos (recall) pero también las falsas alarmas (precisión baja): la curva
  **precisión–recall** describe ese compromiso.
- **AP (*average precision*)** = área bajo esa curva (con la precisión "suavizada": en cada punto se toma la mayor precisión
  que se consigue con ese recall o más). Aquí: 0,25×1,00 + 0,25×1,00 + 0,25×0,75 = **0,6875**. (La 5.ª detección no añade recall.)
- **mAP** = media del AP de **todas las clases** (y, en el estándar COCO, también promediada sobre varios umbrales de IoU de 0,5 a 0,95:
  *mAP@[.5:.95]*). Es la cifra con la que se comparan detectores.

- **Qué hace:**
    - `np.cumsum(es_tp)` acumula los verdaderos positivos y `np.cumsum(1 - es_tp)` los falsos positivos; `precision = tp / (tp + fp)` y `recall = tp / n_reales` en cada puesto.
    - `np.maximum.accumulate(precision[::-1])[::-1]` hace la precisión **monótona** (cada valor pasa a ser el máximo de los que tiene a su derecha).
    - `recall_previo` es el recall del puesto anterior; el AP es la suma de `(recall − recall_previo) × precision`. Devuelve 0 si no hay detecciones o no hay objetos reales.
- **Qué debe salir:**
    - `✔ Ejercicio 3 correcto`.
- **Cómo leerlo:**
    - Con la tabla del enunciado ([1, 1, 0, 1, 0] y 4 objetos reales) el AP vale **0,6875**. Un detector perfecto da 1; uno que solo encuentra la mitad de los objetos, 0,5.
- **Si falla:**
    - `Con la tabla del enunciado el AP debe ser 0.6875` → falta suavizar la precisión de derecha a izquierda, o ponderas mal por el aumento de recall.
    - `Solo encuentra la mitad de los objetos: AP = 0.5` → divides por el número de detecciones en vez de `n_reales`.
    - `NameError: name 'np' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto vale el AP de la tabla del enunciado?* → **0,6875** = 0,25×1,00 + 0,25×1,00 + 0,25×0,75.
2. *¿Por qué la 5.ª detección no añade nada al AP?* → Porque es un falso positivo: **no aumenta el recall** (la suma pondera por el aumento de recall).
3. *¿Qué mide el mAP?* → La **media del AP de todas las clases** (y en COCO, también promediada sobre varios umbrales de IoU de 0,5 a 0,95).

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def average_precision(es_tp, n_reales):
    es_tp = np.asarray(es_tp, dtype=float)
    if len(es_tp) == 0 or n_reales == 0:
        return 0.0
    tp = np.cumsum(es_tp)
    fp = np.cumsum(1 - es_tp)
    precision = tp / (tp + fp)
    recall = tp / n_reales
    precision = np.maximum.accumulate(precision[::-1])[::-1]
    recall_previo = np.concatenate([[0.0], recall[:-1]])
    return float(np.sum((recall - recall_previo) * precision))
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 5. Un detector clásico con OpenCV

#### A5 · Ejemplo — Generar una escena de playa con sombrillas

**Escena.** Vista aérea esquemática de la playa de un resort (128×128 píxeles): arena, franja de mar, **sombrillas**
(círculos rojos, naranjas, amarillos o de un amarillo pastel poco saturado, a veces solapados) y **distractores** (palmeras verdes, tumbonas grises). El generador también entrega
las **cajas reales** y la **máscara** de sombrillas. Son dibujos, no fotos.

- **Qué hace:**
    - `escena(rng)` dibuja una vista aérea de 128×128 píxeles: arena, una franja de mar, sombrillas (círculos de color vivo, a veces solapados) y distractores (palmeras verdes y tumbonas grises). Devuelve la imagen, las **cajas reales** (`[x1, y1, x2, y2]`) y la **máscara** booleana de las sombrillas.
    - `dibujar_cajas(eje, cajas, color, etiquetas)` pinta rectángulos sobre una imagen.
    - Se genera la primera escena (`img0`, con semilla 42) y se muestra sola y con sus cajas reales en verde.
- **Qué debe salir:**
    - Dos imágenes: la escena sintética y la misma con las cajas reales (el título dice `Cajas reales (4 sombrillas)`).
- **Cómo leerlo:**
    - Esta escena tiene **4 sombrillas** reales. Las cajas reales y la máscara nos permitirán medir cualquier detector.
- **Si falla:**
    - `NameError: name 'np' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas sombrillas tiene la primera escena?* → **4**.
2. *¿Qué devuelve `escena` además de la imagen?* → Las **cajas reales** y la **máscara** de sombrillas.
3. *¿Para qué sirven los distractores (palmeras, tumbonas)?* → Para que el detector tenga que **distinguir** las sombrillas de otros objetos.

</details>

#### A6 · Ejemplo — Detector 1: template matching + NMS

Se desliza una **plantilla** (un círculo saturado) por la imagen y se mide el parecido en cada posición
(`cv2.matchTemplate`). Cada sombrilla aparece con **muchas posiciones casi iguales** que superan el umbral: justo el caso
para el que sirve el NMS. Trabajamos sobre el canal de **saturación** (los colores vivos destacan sobre la arena y no dependen del tono).

- **Qué hace:**
    - `saturacion(img)` convierte a HSV y se queda con el canal de saturación (los colores vivos destacan sobre la arena).
    - `plantilla` es un círculo naranja de 24×24 convertido a saturación. `detectar_plantilla(img, umbral)` la desliza por la imagen con `cv2.matchTemplate` y propone una caja de 16×16 en cada posición con parecido ≥ 0,6.
    - `nms(..., umbral_iou=0.3)` (tu función) elimina los duplicados. Se dibujan las cajas antes y después de NMS (rojo = detecciones, verde = reales).
- **Qué debe salir:**
    - `Detecciones brutas: 47 → tras NMS: 6 (sombrillas reales: 4)` y dos imágenes con las cajas antes y después de NMS.
- **Cómo leerlo:**
    - Cada sombrilla genera **muchas cajas casi iguales** (47 en total); NMS las reduce a 6. Siguen siendo más que las 4 reales: algunas cajas son falsos positivos, pero ya no hay duplicados masivos.
- **Si falla:**
    - `NameError: name 'nms' is not defined` → falta el ejercicio 2.
    - `NameError: name 'img0' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas detecciones hay antes y después de NMS?* → **47** antes y **6** después (hay 4 sombrillas reales).
2. *¿Por qué se trabaja sobre el canal de saturación?* → Porque los **colores vivos** destacan sobre la arena y no dependen del tono.
3. *¿Qué muestra este ejemplo sobre el uso de NMS?* → Que el detector da **muchas cajas casi iguales** por objeto y NMS las limpia.

</details>

#### A7 · Ejemplo — Detector 2: color + contornos

Otra idea clásica: (1) pasar a HSV, (2) quedarse con los píxeles de color vivo y tono rojo–amarillo (**máscara**),
(3) limpiar el ruido con una **apertura morfológica**, (4) encontrar los **contornos** (`cv2.findContours`) y (5) quedarse con
la caja que rodea a cada contorno de tamaño razonable. La confianza será lo "circular" que es el contorno.

- **Qué hace:**
    - `detectar_color(img, sat_min, area_min)` pasa a HSV y crea una **máscara** con los píxeles de tono rojo–amarillo, saturados (`s >= sat_min`) y claros (`v >= 120`).
    - `cv2.morphologyEx(..., MORPH_OPEN, ...)` (apertura morfológica) limpia el ruido; `cv2.findContours` encuentra los contornos y se descartan los de área menor que `area_min`.
    - Cada contorno da una caja (`cv2.boundingRect`) y una confianza: lo circular que es (área / área de la elipse inscrita, máximo 1). Devuelve cajas, confianzas y la máscara.
- **Qué debe salir:**
    - Tres imágenes: la escena, la máscara de color vivo y las cajas del detector (rojo, con su confianza) frente a las reales (verde). En esta escena el detector propone **2 cajas** frente a las 4 sombrillas reales.
- **Cómo leerlo:**
    - Se le escapan algunas sombrillas: un detector sencillo de color no las encuentra todas. Cuánto, lo mediremos con las métricas en el siguiente ejemplo.
- **Si falla:**
    - `NameError: name 'cv2' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué hace la apertura morfológica?* → **Elimina el ruido**: borra manchas pequeñas de la máscara.
2. *¿Cómo se calcula la confianza de cada caja?* → Como lo **circular y relleno** que es el contorno: `área / (π/4 · ancho · alto)`, con máximo 1.
3. *¿Cuántas cajas propone para la primera escena y cuántas sombrillas hay?* → **2** cajas frente a **4** sombrillas reales: se le escapan algunas.

</details>

#### A8 · Ejemplo — Evaluar el detector: emparejar con la realidad

Para calcular precisión, recall y AP hay que decidir qué detecciones son verdaderos positivos: se recorren **por confianza
decreciente** y cada una se empareja con la caja real **libre** de mayor IoU, si esta supera el umbral. Una caja real solo puede
emparejarse una vez (las duplicadas cuentan como falsos positivos).

- **Qué hace:**
    - `emparejar(cajas_det, conf, cajas_reales, umbral_iou)` recorre las detecciones por confianza decreciente y marca como verdadero positivo (1) las que se emparejan con una caja real **libre** de IoU ≥ umbral; las demás son falsos positivos (0).
    - `evaluar_detector(detector, n_escenas, semilla, umbral_iou)` aplica el detector a 30 escenas nuevas y calcula el AP, la precisión y el recall globales (con tu `average_precision`).
    - `detector_color` envuelve a `detectar_color` para devolver solo cajas y confianzas.
- **Qué debe salir:**
    - `{'AP': 0.537, 'precisión': 0.887, 'recall': 0.55, 'reales': 171, 'detectadas': 106}`.
- **Cómo leerlo:**
    - De las 106 cajas que da el detector, el 88,7 % es acertado (precisión alta), pero solo encuentra el 55 % de las 171 sombrillas (recall bajo): es un detector **prudente** que se deja casi la mitad de los objetos. El AP resume ambas cosas: 0,537.
- **Si falla:**
    - `NameError: name 'average_precision' is not defined` → falta el ejercicio 3.
    - `NameError: name 'escena' is not defined` → faltan los ejemplos anteriores.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué precisión y qué recall tiene el detector de color?* → Precisión **0,887** y recall **0,55**.
2. *¿Qué significa ese recall?* → Que solo encuentra el **55 %** de las sombrillas reales (94 de 171).
3. *¿Por qué una caja real no puede emparejarse dos veces?* → Para que **las cajas duplicadas cuenten como falsos positivos**, igual que en una evaluación real.

</details>

#### A9 · Ejercicio 4 — El mAP según el umbral de IoU

- **Qué hace:**
    - `ap_por_umbral` guarda el AP del detector de color con cada umbral de IoU (0,5; 0,6; 0,7; 0,8; 0,9), llamando a `evaluar_detector(detector_color, umbral_iou=u)["AP"]`.
    - `map_iou` es la **media** de esos 5 AP (`np.mean`); como solo hay una clase (sombrilla), el mAP coincide con el AP.
    - La comprobación verifica que hay 5 valores y que el AP **no sube** al exigir más IoU, y dibuja la curva.
- **Qué debe salir:**
    - `✔ Ejercicio 4 correcto` y una curva decreciente: AP ≈ 0,54 · 0,51 · 0,48 · 0,29 · 0,00; media (mAP) ≈ 0,36.
- **Cómo leerlo:**
    - Cuanto más exigimos a las cajas, peor sale: con IoU ≥ 0,5 el AP es 0,54 y con IoU ≥ 0,9 cae a prácticamente 0. El detector **encuentra** las sombrillas, pero sus cajas no son precisas.
- **Si falla:**
    - `Debe haber un AP por cada umbral (5 en total)` → el bucle no recorre los 5 umbrales.
    - `El AP no debería subir al exigir más IoU. Revisa el orden de los umbrales` → los umbrales se están recorriendo en otro orden.
    - `NameError: name 'detector_color' is not defined` → falta el ejemplo de evaluar el detector.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto vale el mAP del detector de color con estos umbrales?* → **≈ 0,36** (la media de 0,54; 0,51; 0,48; 0,29 y 0,00).
2. *¿Por qué el AP baja al subir el umbral de IoU?* → Porque se exige que **las cajas coincidan mucho más** con las reales; las nuestras son aproximadas.
3. *¿Qué indica un AP casi 0 con IoU ≥ 0,9?* → Que las cajas **no encajan con precisión**, aunque el detector encuentre el objeto.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
umbrales = [0.5, 0.6, 0.7, 0.8, 0.9]      # los umbrales de IoU que vas a probar (ya escritos)
ap_por_umbral = [evaluar_detector(detector_color, umbral_iou=u)["AP"] for u in umbrales]
map_iou = float(np.mean(ap_por_umbral))


plt.plot(umbrales, ap_por_umbral, "o-"); plt.xlabel("umbral de IoU exigido"); plt.ylabel("AP")
plt.title(f"Detector de color: AP según el umbral de IoU (media = {map_iou:.2f})"); plt.ylim(0, 1.02); plt.show()
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 6. Segmentación

#### A10 · Ejercicio 5 — IoU y Dice de máscaras

**Idea.** Ahora no buscamos cajas sino una **máscara**: para cada píxel, ¿es sombrilla o no? La métrica natural es
la IoU de máscaras (o el **coeficiente Dice**, muy usado en medicina):

- `IoU = |A ∩ B| / |A ∪ B|` · `Dice = 2·|A ∩ B| / (|A| + |B|)`

La máscara que produce nuestro detector de color ya es una segmentación: la comparamos con la máscara real.

- **Qué hace:**
    - `iou_mascara(pred, real)` calcula `|A ∩ B| / |A ∪ B|` contando los píxeles `True` de `np.logical_and` y de `np.logical_or`; devuelve 1.0 si ambas máscaras están vacías.
    - `dice(pred, real)` calcula `2·|A ∩ B| / (|A| + |B|)` (con 1.0 si ambas están vacías).
    - La comprobación usa dos máscaras 2×3 de ejemplo (`_p` y `_r`): intersección 1 y unión 3 → IoU 1/3; Dice 2·1/(2+2) = 0,5.
- **Qué debe salir:**
    - `✔ Ejercicio 5 correcto`.
- **Cómo leerlo:**
    - El Dice siempre vale **igual o más** que la IoU para las mismas máscaras (aquí 0,5 frente a 0,33): da más peso a la intersección.
- **Si falla:**
    - `Intersección 1, unión 3 → 1/3` → la unión o la intersección están mal contadas (usa `.sum()` de las máscaras booleanas).
    - `2·1 / (2 + 2) = 0.5` → en `dice` falta el factor 2 o divides entre la unión.
    - `Dos máscaras vacías coinciden: 1.0` → falta el caso de división por 0.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto valen IoU y Dice de `_p` y `_r`?* → IoU **1/3** y Dice **0,5**.
2. *¿Por qué se devuelve 1.0 si ambas máscaras están vacías?* → Porque **coinciden** (no hay nada que fallar) y evitan dividir entre cero.
3. *¿Qué diferencia hay entre IoU y Dice?* → La IoU divide la intersección entre la **unión**; el Dice divide el doble de la intersección entre la **suma de áreas**: es siempre mayor o igual.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def iou_mascara(pred, real):
    inter = np.logical_and(pred, real).sum()
    union = np.logical_or(pred, real).sum()
    return float(inter / union) if union > 0 else 1.0

def dice(pred, real):
    inter = np.logical_and(pred, real).sum()
    total = pred.sum() + real.sum()
    return float(2 * inter / total) if total > 0 else 1.0
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A11 · Ejemplo — Segmentar con el umbral de color

- **Qué hace:**
    - Para 20 escenas nuevas (`default_rng(200)`), `detectar_color(im)[2]` da la máscara del detector de color (`pred`) y se compara con la máscara real con `iou_mascara` y `dice`.
    - El `print` muestra la media de ambas métricas.
- **Qué debe salir:**
    - `Segmentación por umbral de color: IoU media = 0.629 · Dice medio = 0.756`.
- **Cómo leerlo:**
    - La máscara del detector de color ya es una segmentación: solapa de media el **63 %** (IoU) con la máscara real. Sin entrenar ningún modelo.
- **Si falla:**
    - `NameError: name 'iou_mascara' is not defined` → falta el ejercicio 5.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué IoU y Dice medios logra la segmentación por color?* → **0,629** y **0,756**.
2. *¿Por qué el Dice sale mayor que la IoU?* → Porque da **más peso a la intersección** (es siempre mayor o igual).
3. *¿Hace falta entrenar algo para obtener esa máscara?* → **No**: es una regla de color más una limpieza morfológica.

</details>

#### A12 · Ejemplo — Segmentar por agrupación de colores (k-means)

Sin entrenar ningún modelo: se agrupan los píxeles en *k* colores. Probamos con una foto de ejemplo que trae scikit-learn
(`load_sample_images`, incluida en la instalación: no necesita internet).

- **Qué hace:**
    - `load_sample_images().images[1]` carga una foto de una flor incluida en `scikit-learn` (sin internet) y `cv2.resize` la reduce a 160×107.
    - `cv2.kmeans(..., 4, ...)` agrupa los píxeles en 4 colores; cada píxel se sustituye por el color de su grupo (`segmentada`).
    - Se muestran la foto original y la segmentada.
- **Qué debe salir:**
    - Dos imágenes: la foto de la flor y la misma imagen reducida a 4 colores planos.
- **Cómo leerlo:**
    - Sin entrenar ningún modelo, k-means **agrupa los píxeles por color** y separa zonas (pétalos, hojas, fondo); no sabe qué son: solo agrupa colores parecidos.
- **Si falla:**
    - `ModuleNotFoundError` en `sklearn` → falta `scikit-learn` en el entorno.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántos colores quedan en la imagen segmentada?* → **4**, uno por grupo.
2. *¿Sabe k-means qué es cada zona (flor, hoja…)?* → **No**: solo agrupa píxeles de color parecido.
3. *¿Cuándo funciona bien esta idea?* → Cuando los objetos tienen **colores muy distintos** del fondo.

</details>

#### A13 · Ejemplo — Una mini red de segmentación (encoder-decoder)

Las redes modernas de segmentación (U-Net, DeepLab) tienen dos mitades: un **codificador** que reduce la imagen extrayendo
rasgos (convolución + pooling) y un **decodificador** que la vuelve a agrandar (`UpSampling`) hasta dar **una probabilidad por
píxel**. Aquí una versión diminuta entrenada en 300 escenas sintéticas para "pintar" las sombrillas.

- **Qué hace:**
    - Se generan 360 escenas: 300 para entrenar y 60 para probar, con sus máscaras reales (`X_seg` y `Y_seg`).
    - `construir_segmentador()` crea un **codificador** (`Conv2D` + `MaxPooling2D`, que reduce a 64×64) y un **decodificador** (`UpSampling2D`, que vuelve a 128×128) que acaba en una `Conv2D` con sigmoide: una probabilidad por píxel. Se entrena 12 épocas con `binary_crossentropy`.
    - Se calcula la IoU media de las máscaras predichas y se compara con la del detector de color; se muestran imagen, máscara real y predicción de la primera escena de prueba.
- **Qué debe salir:**
    - `IoU de máscaras en 60 escenas nuevas → mini red: 0.612 · umbral de color: 0.587` y tres imágenes (imagen, máscara real y predicción).
- **Cómo leerlo:**
    - Los dos métodos están **parejos** (0,61 frente a 0,59): en un problema tan limpio (color inconfundible), un método clásico bien pensado es un buen punto de partida. Las redes ganan cuando los objetos varían mucho de aspecto, hay sombras, oclusiones o fondos complejos.
- **Si falla:**
    - `NameError: name 'detectar_color' is not defined` → faltan los ejemplos anteriores. Tarda unos segundos en entrenar.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué IoU logran la mini red y el umbral de color?* → **0,612** y **0,587**.
2. *¿Qué hace cada mitad de una red encoder-decoder?* → El **codificador** reduce la imagen extrayendo rasgos; el **decodificador** la agranda hasta dar **una probabilidad por píxel**.
3. *¿Cuándo preferirías la red al umbral de color?* → Cuando los objetos **varían mucho** de aspecto, hay sombras, oclusiones o fondos complejos.

</details>

## 7. Colab (opcional): YOLO

#### A14 · Ejemplo — Un detector moderno con YOLO (opcional, requiere internet)

> **AVISO: esta celda requiere internet y NO se ha verificado en el taller.** Instala la librería `ultralytics`, descarga
> los pesos preentrenados `yolov8n.pt` (≈ 6 MB, entrenados en COCO: 80 clases como persona, coche, silla, paraguas) y una
> imagen de ejemplo. Pon `USAR_INTERNET = True` en Colab. Fíjate en que YOLO devuelve **cajas + clase + confianza** y que
> ya aplica NMS internamente (parámetro `iou`).

**Qué cambia con un detector moderno:** aprende el detector completo a partir de miles de imágenes etiquetadas con cajas
(etiquetar cajas cuesta más que etiquetar clases), predice cientos de cajas por imagen en una sola pasada y se evalúa con
**mAP@[.5:.95]**. Lo que has hecho a mano (IoU, NMS, AP) es exactamente lo que ocurre dentro y alrededor de YOLO.

- **Qué hace:**
    - Con `USAR_INTERNET = False` (por defecto) la celda solo escribe un mensaje. Con `True` instala `ultralytics`, descarga los pesos `yolov8n.pt` (entrenados en COCO) y una imagen de ejemplo, detecta objetos y dibuja las cajas, y escribe la clase, la confianza y la caja de cada detección.
- **Qué debe salir:**
    - `Rama de Colab desactivada (USAR_INTERNET = False). El resto del notebook no la necesita.`
- **Cómo leerlo:**
    - Esta rama **no se ha verificado** sin internet. Para probarla, en Colab, cambia `USAR_INTERNET` a `True`.
- **Si falla:**
    - Con `USAR_INTERNET = True` sin conexión → error de instalación o descarga: déjalo en `False`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué devuelve YOLO por cada objeto?* → Una **caja**, una **clase** y una **confianza**.
2. *¿Qué parte de lo que has programado a mano aparece dentro de YOLO?* → **IoU, NMS y AP**: YOLO aplica NMS internamente (parámetro `iou`) y se evalúa con mAP.
3. *¿Por qué etiquetar para un detector cuesta más que para un clasificador?* → Porque hay que dibujar **cajas** sobre cada objeto, no solo poner una etiqueta por imagen.

</details>

## 8. Mini-reto: contar sombrillas

#### A15 · Mini-reto — Contador de sombrillas

- **Qué hace:**
    - `error_conteo` es la media del valor absoluto de `len(cajas del detector) - len(reales)` en 30 escenas nuevas (`default_rng(500)`).
    - Para cada `sat_min` (60, 90, 120, 150, 180), `evaluar_detector(lambda im, s=s: detectar_color(im, sat_min=s)[:2], ...)` da la precisión `p` y el recall `rec`, y `f1_por_sat[s] = 2·p·rec / (p + rec)`.
    - `mejor_sat` es el `sat_min` con mayor F1 (`max(f1_por_sat, key=f1_por_sat.get)`) y `f1_mejor` su valor.
- **Qué debe salir:**
    - `Error medio de conteo: 1.87 sombrillas por foto · mejor sat_min = 90 (F1 = 0.71)`, `{60: 0.08, 90: 0.71, 120: 0.69, 150: 0.69, 180: 0.69}` y `✔ Mini-reto correcto`.
- **Cómo leerlo:**
    - Con `sat_min = 60` la arena «cuela» como sombrilla: se detectan 225 cajas para 165 sombrillas reales (precisión 0,07) y el F1 se hunde a 0,08. De 120 en adelante el F1 es el mismo (0,69): el detector no encuentra más sombrillas, así que la precisión (0,87) y el recall (0,58) no cambian. El mejor es **90** (precisión 0,77 y recall 0,65). Por término medio el contador se equivoca en unas **1,9 sombrillas por foto**.
- **Si falla:**
    - `El error de conteo no puede ser negativo` → has restado sin valor absoluto.
    - `mejor_sat debe ser uno de los valores probados` → no es una de las claves de `f1_por_sat`.
    - `Con el mejor umbral el F1 debería superar 0.5` → precisión o recall mal calculados (usa `evaluar_detector`).
    - `NameError: name 'evaluar_detector' is not defined` → faltan los ejemplos anteriores.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué `sat_min` da el mejor F1 y cuánto vale?* → **90**, con F1 **0,71**.
2. *¿Por qué falla con `sat_min = 60`?* → Porque con un umbral tan bajo **la arena cuela como sombrilla**: muchas falsas detecciones (precisión 0,07).
3. *¿Por qué el F1 no cambia entre 120 y 180?* → Porque el detector ya solo se queda con los colores muy saturados: **no encuentra más sombrillas** (recall 0,58) y la precisión no cambia (0,87).
4. *¿Qué pasaría con sombrillas blancas?* → El detector de color **no las vería**: el blanco tiene saturación casi cero. Habría que usar otra señal (forma, plantilla) o un modelo entrenado.
5. *¿Qué harías para que funcione con fotos reales?* → Entrenar o ajustar un **detector moderno** (por ejemplo YOLO) con fotos reales etiquetadas con cajas, y evaluarlo con mAP en fotos con distintas luces, sombras y sombrillas pegadas.

</details>

## 9. Preguntas de reflexión
1. Un detector con AP = 0,9 a IoU 0,5 pero 0,4 a IoU 0,9: ¿qué te dice sobre la calidad de sus cajas?
2. ¿Por qué sin NMS un detector cuenta varias veces el mismo objeto? ¿Qué pasaría con un umbral de NMS demasiado bajo si hay objetos muy juntos?
3. ¿Cuándo elegirías segmentación en vez de detección para TurisData (superficie de piscina, ocupación de la playa…)?
4. Etiquetar cajas es más caro que etiquetar clases. ¿Cómo influiría en la decisión de usar YOLO?

<details><summary>Ver respuestas (después de pensarlo)</summary>

1. Que **encuentra bien los objetos** (a IoU 0,5 casi todo cuenta como acierto) pero sus **cajas no son precisas**: se desajustan de los bordes y no superan el listón exigente de IoU 0,9. Nuestro detector de color es un ejemplo: AP 0,54 a IoU 0,5 y casi 0 a IoU 0,9.
2. Porque el detector propone **muchas cajas casi iguales** por objeto (47 cajas para 4 sombrillas) y cada una se contaría como una detección. Con un umbral de NMS **demasiado bajo**, dos objetos muy juntos tienen cajas que se solapan y NMS **borraría una** de ellas: se pierden objetos (baja el recall).
3. Cuando interesa el **área o la forma** y no el número de unidades: superficie de piscina, porcentaje de playa ocupada, zonas verdes. La detección se prefiere para **contar** objetos (sombrillas, tumbonas).
4. Entrenar un YOLO propio exige **muchas imágenes con cajas dibujadas**, y eso es caro. Con pocos datos conviene un modelo **preentrenado** (COCO) con *fine-tuning*, un detector clásico como punto de partida, o plantearse si basta con clasificar.

</details>

## 10. Resumen: qué has aprendido
- Clasificar = qué; detectar = qué y dónde (cajas); segmentar = qué es cada píxel.
- **IoU** mide el solape; **NMS** elimina duplicados; **AP/mAP** resumen el compromiso precisión–recall (y mAP@[.5:.95] exige cajas precisas).
- Un detector clásico (color + contornos, *template matching*) sirve de referencia y se evalúa con las mismas métricas que uno moderno.
- La segmentación se mide con IoU/Dice de máscaras; un encoder-decoder diminuto ya aprende a pintar objetos sencillos.
- YOLO automatiza todo el proceso con redes profundas, a cambio de necesitar muchas imágenes etiquetadas con cajas.